# 03 — ADME Data Quantity Study (Phases 1-3: Coarse Scan, Collapse ID, Zoom Sweep)

**Goal**: Find where RF / LightGBM / FCNN performance collapses as training data shrinks,
on the ADME `hybrid` featureset (FCFP4 + normalized RDKit2D descriptors) -- the
primary analyzed featureset -- for all four modelling endpoints: HLM, SOL, MDR1, RLM
(train pools range 1737-2468 rows; PPB_H/PPB_R remain excluded from modelling per
DECISIONS.md).

**Design** (see conversation record / DECISIONS.md for the full rationale):
1. **Phase 1** (§3-4 below) — coarse untuned scan: log-spaced fractions, few seeds,
   baseline (default hyperparameter) arm only. Purpose is purely to locate the collapse
   region, not to produce a final publishable curve.
2. **Phase 2** (§5 below) — identify the collapse point per (endpoint, model): a fixed
   threshold rule plus visual confirmation against the plotted curves.
3. **Phase 3** (§6-7 below) — zoom in around the identified knee with finer fractions and
   more seeds.
4. **Phase 4** (separate notebook/later cells) — light hyperparameter tuning, applied ONLY
   at the identified collapse point(s), not across the whole grid.

**Metrics**: R² (primary — sensitive to the collapse-to-mean failure mode) and MAE
(secondary — orthogonal magnitude check, immune to R²'s variance-driven instability at
small N). Pearson r / Spearman / CCC are also recorded (via `evaluate_model`) but not
plotted here — see conversation record for why R²+MAE were chosen over them.

**Noise integration (future)**: the results schema below already carries `noise_type` /
`noise_level` columns (fixed to `'none'` / `0.0` in this notebook) so a later noise study
can reuse the exact same checkpointed-eval loop and results file — no schema change needed.

**Featureset note**: both `fcfp4` and `hybrid` are computed and checkpointed (§1-3) --
`ANALYSIS_FEATURESET = 'hybrid'` (§2) selects which one §4 onward analyzes and plots;
`fcfp4` results remain in the CSV for reference only.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from sklearn.base import clone
from sklearn.preprocessing import RobustScaler

from src.models import get_paper_models, evaluate_model, run_checkpointed_eval
from src.plotting import ENDPOINT_LABELS

SEED = 42
DATA_PROC = '../data/processed'
FIGURES = '../figures'
os.makedirs(FIGURES, exist_ok=True)

print('Imports OK')

## 1 — Load existing splits (fcfp4 + hybrid)

Reuses the `(endpoint, featureset)` splits already computed and checkpointed by
`01.5_adme_biogen_public_recreation.ipynb` §4.1 — no refeaturization. Each entry carries
`X_train`/`X_test` (raw feature arrays for each featureset, for RF/LightGBM) and `y_train`/`y_test`. FCNN also uses
raw `X_train`/`X_test` here — its `RobustScaler` is fit fresh on each subsample in §3 below
(not the pre-scaled full-pool version from 01.5), so scaling statistics match the data
volume the model actually sees at each fraction instead of leaking full-pool stats into
small-N fits.

In [ ]:
EPS = ['HLM', 'SOL', 'MDR1', 'RLM']
FEATURESETS = ['fcfp4', 'hybrid']  # hybrid = FCFP4 + 316 RDKit descriptors (DECISIONS.md); fcfp4
                                    # kept alongside for the already-computed baseline, not dropped.

splits = joblib.load(f'{DATA_PROC}/section4_splits.pkl')
ep_data = {ep: splits[(ep, 'fcfp4')] for ep in EPS}  # row counts only (same n_train/n_test across featuresets)
ep_data_fs = {(ep, fs): splits[(ep, fs)] for ep in EPS for fs in FEATURESETS}

for ep in EPS:
    for fs in FEATURESETS:
        split = ep_data_fs[(ep, fs)]
        print(f"  {ep}/{fs}: X_train={split['X_train'].shape}, X_test={split['X_test'].shape}")

## 2 — Config

In [ ]:
FRACTIONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
N_SEEDS_COARSE = 5
MODELS = ['RF', 'LightGBM', 'FCNN']
SCALED_MODELS = {'FCNN'}  # models that need RobustScaler'd X
ANALYSIS_FEATURESET = 'hybrid'  # what §4 onward analyzes/plots; 'fcfp4' stays in the CSV for reference

# Placeholder noise columns -- fixed for this quantity-only pass. A later noise study reuses
# this exact schema/checkpoint with noise_type/noise_level varying instead of fixed.
NOISE_TYPE, NOISE_LEVEL = 'none', 0.0
ARM = 'base'  # untuned -- Phase 1 is baseline-only, no tuning arm here

print(f'{len(EPS)} endpoints x {len(FEATURESETS)} featuresets x {len(MODELS)} models x '
      f'{len(FRACTIONS)} fractions x {N_SEEDS_COARSE} seeds = '
      f'{len(EPS) * len(FEATURESETS) * len(MODELS) * len(FRACTIONS) * N_SEEDS_COARSE} fits '
      f'(fcfp4 keys already checkpointed -- only hybrid computes fresh)')

## 3 — Coarse sweep

Checkpointed via `run_checkpointed_eval` — re-running this cell only computes missing
`(endpoint, model, fraction, seed, noise_type, noise_level, arm)` keys, so it's safe to
interrupt or extend `FRACTIONS`/`N_SEEDS_COARSE` later without recomputing everything.

In [ ]:
RESULTS_CSV = f'{DATA_PROC}/quantity_scan_results.csv'
PREDICTIONS_PKL = f'{DATA_PROC}/quantity_scan_predictions.pkl'
KEY_COLS = ('endpoint', 'featureset', 'model', 'fraction', 'seed', 'noise_type', 'noise_level', 'arm')

paper_models = get_paper_models()

def compute_one(key):
    ep, fs, model_name, frac, seed, noise_type, noise_level, arm = key
    split = ep_data_fs[(ep, fs)]
    scaled = model_name in SCALED_MODELS
    X_train_full = split['X_train']
    X_test = split['X_test']
    y_train_full, y_test = split['y_train'], split['y_test']

    n_sub = max(round(len(X_train_full) * frac), 2)
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X_train_full), n_sub, replace=False)
    X_sub, y_sub = X_train_full[idx], y_train_full[idx]

    if scaled:
        # Fit on THIS subsample, not the full pool -- so scaling stats reflect the data
        # volume the model actually trains on at each fraction (see §1).
        scaler = RobustScaler().fit(X_sub)
        X_sub = scaler.transform(X_sub)
        X_test = scaler.transform(X_test)

    model = clone(paper_models[model_name])
    model.fit(X_sub, y_sub)
    y_pred = model.predict(X_test)
    metrics = evaluate_model(None, None, y_test, y_pred=y_pred)

    row = {
        'endpoint': ep, 'featureset': fs, 'model': model_name, 'fraction': frac, 'n_train': n_sub,
        'seed': seed, 'noise_type': noise_type, 'noise_level': noise_level, 'arm': arm,
        **metrics,
    }
    pred = {'y_test': y_test, 'y_pred_test': y_pred}
    return row, pred

keys = [
    (ep, fs, model_name, frac, seed, NOISE_TYPE, NOISE_LEVEL, ARM)
    for ep in EPS
    for fs in FEATURESETS
    for model_name in MODELS
    for frac in FRACTIONS
    for seed in range(N_SEEDS_COARSE)
]

scan_results, scan_predictions = run_checkpointed_eval(
    keys, compute_one, RESULTS_CSV, PREDICTIONS_PKL, key_cols=KEY_COLS,
)
print(scan_results.shape)
scan_results.head()

## 4 — Plot: R² and MAE vs training set size

In [ ]:
# Distinct, colorblind-safe colors -- blue/red/orange is easier to tell apart than
# blue/red/purple, which read too similarly.
MODEL_COLORS = {'RF': '#1f77b4', 'LightGBM': '#d62728', 'FCNN': '#ff9900'}

coarse_title = (
    f'Data quantity coarse scan -- {" / ".join(MODELS)} on {ANALYSIS_FEATURESET}\n'
    f'shaded: 95% CI, n={N_SEEDS_COARSE} seeds/point'
)

fig, axes = plt.subplots(3, len(EPS), figsize=(6 * len(EPS), 12), sharex=False)

for col, ep in enumerate(EPS):
    ep_df = scan_results[(scan_results['endpoint'] == ep) & (scan_results['featureset'] == ANALYSIS_FEATURESET)]
    full_n = ep_data[ep]['X_train'].shape[0]  # 100% of the training pool for this endpoint (featureset-independent)

    # One EVENLY SPACED position per tested fraction (not a log-scaled axis) -- avoids the
    # confusing dual log-axis from before. Each point gets a tick labeled with its actual
    # n_train and % of pool directly, so there's nothing to read off a scale.
    frac_order = sorted(ep_df['fraction'].unique())
    x_pos = {frac: i for i, frac in enumerate(frac_order)}

    for model_name in MODELS:
        m_df = ep_df[ep_df['model'] == model_name].groupby('fraction').agg(
            R2_mean=('R2', 'mean'), R2_std=('R2', 'std'), n_seeds=('R2', 'size'),
            MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
            Pearson_mean=('Pearson', 'mean'), Pearson_std=('Pearson', 'std'),
            n_train=('n_train', 'first'),
        ).reset_index().sort_values('fraction')
        # 95% CI via a t-multiplier (not a fixed 1.96) -- N_SEEDS_COARSE=5 is small enough
        # that the normal approximation understates the interval; t accounts for that.
        tcrit = stats.t.ppf(0.975, m_df['n_seeds'] - 1)
        m_df['R2_ci'] = tcrit * m_df['R2_std'] / np.sqrt(m_df['n_seeds'])
        m_df['MAE_ci'] = tcrit * m_df['MAE_std'] / np.sqrt(m_df['n_seeds'])
        m_df['Pearson_ci'] = tcrit * m_df['Pearson_std'] / np.sqrt(m_df['n_seeds'])

        x = m_df['fraction'].map(x_pos)
        color = MODEL_COLORS[model_name]

        ax_r2 = axes[0, col]
        ax_r2.plot(x, m_df['R2_mean'], marker='o', label=model_name, color=color)
        ax_r2.fill_between(x, m_df['R2_mean'] - m_df['R2_ci'], m_df['R2_mean'] + m_df['R2_ci'],
                           alpha=0.25, color=color)

        ax_pearson = axes[1, col]
        ax_pearson.plot(x, m_df['Pearson_mean'], marker='o', label=model_name, color=color)
        ax_pearson.fill_between(x, m_df['Pearson_mean'] - m_df['Pearson_ci'], m_df['Pearson_mean'] + m_df['Pearson_ci'],
                                alpha=0.25, color=color)

        ax_mae = axes[2, col]
        ax_mae.plot(x, m_df['MAE_mean'], marker='o', label=model_name, color=color)
        ax_mae.fill_between(x, m_df['MAE_mean'] - m_df['MAE_ci'], m_df['MAE_mean'] + m_df['MAE_ci'],
                            alpha=0.25, color=color)

    tick_pos = [x_pos[f] for f in frac_order]
    tick_labels = []
    for f in frac_order:
        n = int(ep_df[ep_df['fraction'] == f]['n_train'].iloc[0])
        pct = n / full_n * 100
        tick_labels.append(f'{n}\n({pct:.0f}%)')

    for row in (0, 1, 2):
        ax = axes[row, col]
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_labels)
        ax.set_title(ENDPOINT_LABELS.get(ep, ep), loc='center')
        ax.set_xlabel('n_train (% of training pool)')
        # Seeds/point note on every subplot -- not just col 0 -- so each panel is
        # self-contained when read/exported individually.
        ax.set_title(f'n={N_SEEDS_COARSE} seeds/point', loc='left', fontsize=7)

    axes[0, col].axhline(0, color='grey', linewidth=0.8, linestyle='--')
    axes[0, col].set_ylabel('R² (test)')
    axes[0, col].grid(alpha=0.3)

    axes[1, col].set_ylabel('Pearson r (test)')
    axes[1, col].grid(alpha=0.3)

    axes[2, col].set_ylabel(f'MAE (test) [{ENDPOINT_LABELS.get(ep, ep)}]')
    axes[2, col].grid(alpha=0.3)

    # CI patch appended to every legend -- so each subplot's shading is explained without
    # relying on the reader having seen the figure suptitle.
    ci_patch = Patch(facecolor='grey', alpha=0.25, label='95% CI')
    for row in (0, 1, 2):
        handles, labels = axes[row, col].get_legend_handles_labels()
        axes[row, col].legend(handles=handles + [ci_patch], fontsize=8)

fig.suptitle(coarse_title, fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES}/quantity_scan_coarse.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — Collapse identification (threshold pass)

`fraction_remaining` = the fraction of the training pool **still present** (i.e.
`n_train / full_N`), not the fraction removed. Scanning fractions from 1.0 downward, it's
the first (largest) fraction at which mean R² has dropped more than 0.1 absolute below the
full-N R², or gone negative -- whichever comes first. E.g. `0.20` for HLM/RF means R²
already collapsed by the point only 20% of the training data remained.

Because the threshold is an *absolute* R² drop, it isn't directly comparable across
endpoints/models with different full-N R² (SOL's full-N R² is much lower than HLM's, so the
same 0.1 drop eats a bigger relative share of SOL's signal). `frac_of_full_R2` reports R² at
the collapse point as a fraction of full-N R², making collapse severity comparable across
rows.

This is a starting point only; confirm/adjust against the plot above before locking in
Phase 3 zoom points.

In [ ]:
COLLAPSE_R2_DROP = 0.1

collapse_rows = []
for ep in EPS:
    for model_name in MODELS:
        sub = scan_results[(scan_results['endpoint'] == ep) & (scan_results['featureset'] == ANALYSIS_FEATURESET)
                            & (scan_results['model'] == model_name)]
        by_frac = sub.groupby('fraction')['R2'].mean().sort_index()
        full_r2 = by_frac.loc[1.0]

        # frac here is the fraction of the training pool RETAINED (n_train / full_N),
        # scanned from 1.0 (all data) downward toward 0 (least data).
        collapse_frac = None
        for frac in sorted(by_frac.index, reverse=True):
            if frac == 1.0:
                continue  # full-N point is the reference, not a candidate
            r2 = by_frac.loc[frac]
            if r2 < 0 or (full_r2 - r2) > COLLAPSE_R2_DROP:
                collapse_frac = frac
                break  # first (largest) retained-fraction where R2 has already collapsed

        collapse_r2 = None if collapse_frac is None else by_frac.loc[collapse_frac]
        collapse_rows.append({
            'endpoint': ep, 'model': model_name, 'full_N_R2': full_r2,
            'fraction_remaining': collapse_frac,
            'collapse_n_train': None if collapse_frac is None
                else int(sub[sub['fraction'] == collapse_frac]['n_train'].iloc[0]),
            'collapse_R2': collapse_r2,
            # R2 at the collapse point as a fraction of full-N R2 -- an absolute 0.1 drop eats a much
            # bigger relative share of a low full_N_R2 (e.g. SOL) than a high one (e.g. HLM), so this
            # column is what makes collapse severity comparable ACROSS endpoints/models.
            'frac_of_full_R2': None if collapse_r2 is None else collapse_r2 / full_r2,
        })

collapse_df = pd.DataFrame(collapse_rows)
collapse_df

## 6 — Phase 3: Zoom sweep (finer fractions, more seeds)

The coarse scan (§3) located each model's knee somewhere in the 0.05–0.75 band, but with
only 5 seeds the mean curve there is noisy enough that small non-monotonic wiggles (see
FCNN discussion) can't be distinguished from real trend. This pass uses one shared, finer
fraction grid across all models/endpoints (`ZOOM_FRACTIONS`, range 0.02–1.0) with 15
additional seeds per point -- including 0.75/1.0, so the full-range curve in §7 has a
uniform seed count end to end instead of mixing in the coarse scan's 5-seed points.

Zoom seeds are numbered 5–19 (continuing on from the coarse scan's 0–4), not 0–14, so that
at the fractions the two grids share (0.02, 0.05, 0.1, 0.2, 0.5, 0.75, 1.0) the results
*add* to the coarse scan's seeds (20 total combined) rather than recomputing the same 5
redundantly. Fractions unique to the zoom grid (0.08, 0.15, 0.25, 0.3, 0.4, 0.6) get 15
seeds of their own.

Checkpointed the same way as §3 — safe to interrupt/resume. At ~7.8s/fit for FCNN (the
slowest of the three models), the full sweep takes on the order of an hour per featureset.

In [ ]:
ZOOM_FRACTIONS = [0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0]
N_SEEDS_ZOOM = 15
ZOOM_SEED_OFFSET = N_SEEDS_COARSE  # start at 5, not 0 -- see §6 markdown

ZOOM_RESULTS_CSV = f'{DATA_PROC}/quantity_scan_zoom_results.csv'
ZOOM_PREDICTIONS_PKL = f'{DATA_PROC}/quantity_scan_zoom_predictions.pkl'

zoom_keys = [
    (ep, fs, model_name, frac, seed, NOISE_TYPE, NOISE_LEVEL, ARM)
    for ep in EPS
    for fs in FEATURESETS
    for model_name in MODELS
    for frac in ZOOM_FRACTIONS
    for seed in range(ZOOM_SEED_OFFSET, ZOOM_SEED_OFFSET + N_SEEDS_ZOOM)
]

print(f'{len(EPS)} endpoints x {len(FEATURESETS)} featuresets x {len(MODELS)} models x '
      f'{len(ZOOM_FRACTIONS)} fractions x {N_SEEDS_ZOOM} seeds = {len(zoom_keys)} fits '
      f'(fcfp4/0.02-0.6 keys already checkpointed -- only hybrid and the new 0.75/1.0 fractions '
      f'compute fresh)')

In [ ]:
zoom_results, zoom_predictions = run_checkpointed_eval(
    zoom_keys, compute_one, ZOOM_RESULTS_CSV, ZOOM_PREDICTIONS_PKL, key_cols=KEY_COLS,
)
print(zoom_results.shape)
zoom_results.head()

## 6.1 — Collapse identification (zoom, finer grid + more seeds)

Same threshold rule as §5, re-run on `zoom_results` instead of `scan_results` -- 15 seeds/point
on the finer `ZOOM_FRACTIONS` grid (vs. 5 seeds on the coarse `FRACTIONS` grid), so the
collapse point located here is less sensitive to seed noise than the §5 pass. `full_r2` is
still read from the coarse scan's `fraction == 1.0` row (rather than the zoom grid's own 1.0
point) purely for continuity with §5's numbers -- both are the same fraction, just from
different seed sets.

In [ ]:
collapse_rows_zoom = []
for ep in EPS:
    for model_name in MODELS:
        full_sub = scan_results[(scan_results['endpoint'] == ep) & (scan_results['featureset'] == ANALYSIS_FEATURESET)
                                 & (scan_results['model'] == model_name)]
        full_r2 = full_sub.groupby('fraction')['R2'].mean().loc[1.0]

        sub = zoom_results[(zoom_results['endpoint'] == ep) & (zoom_results['featureset'] == ANALYSIS_FEATURESET)
                            & (zoom_results['model'] == model_name)]
        by_frac = sub.groupby('fraction')['R2'].mean().sort_index()

        # frac here is the fraction of the training pool RETAINED (n_train / full_N),
        # scanned from the largest zoom fraction downward toward the smallest.
        collapse_frac = None
        for frac in sorted(by_frac.index, reverse=True):
            r2 = by_frac.loc[frac]
            if r2 < 0 or (full_r2 - r2) > COLLAPSE_R2_DROP:
                collapse_frac = frac
                break  # first (largest) retained-fraction where R2 has already collapsed

        collapse_r2 = None if collapse_frac is None else by_frac.loc[collapse_frac]
        collapse_rows_zoom.append({
            'endpoint': ep, 'model': model_name, 'full_N_R2': full_r2,
            'fraction_remaining': collapse_frac,
            'collapse_n_train': None if collapse_frac is None
                else int(sub[sub['fraction'] == collapse_frac]['n_train'].iloc[0]),
            'collapse_R2': collapse_r2,
            'frac_of_full_R2': None if collapse_r2 is None else collapse_r2 / full_r2,
        })

collapse_df_zoom = pd.DataFrame(collapse_rows_zoom)
collapse_df_zoom

## 7 — Zoom curve (0.02–1.0, 15 seeds/point)

R² / Pearson r / MAE vs. training-set size, plotted from `zoom_results`, spans the 0.02–1.0 range at a uniform 15 seeds/point. 

In [ ]:
# zoom_results only, filtered to the primary featureset -- ZOOM_FRACTIONS now spans 0.02-1.0
# with a uniform 15 seeds/point (see §6), so no coarse-scan splicing needed.
plot_results = zoom_results[zoom_results['featureset'] == ANALYSIS_FEATURESET]

zoom_title = (
    f'Data quantity: zoom range (0.02-1.0) -- {" / ".join(MODELS)} on {ANALYSIS_FEATURESET}\n'
    f'shaded: 95% CI, n={N_SEEDS_ZOOM} seeds/point'
)

fig, axes = plt.subplots(3, len(EPS), figsize=(6 * len(EPS), 12), sharex=False)

for col, ep in enumerate(EPS):
    ep_df = plot_results[plot_results['endpoint'] == ep]
    full_n = ep_data[ep]['X_train'].shape[0]

    frac_order = sorted(ep_df['fraction'].unique())
    x_pos = {frac: i for i, frac in enumerate(frac_order)}

    for model_name in MODELS:
        m_df = ep_df[ep_df['model'] == model_name].groupby('fraction').agg(
            R2_mean=('R2', 'mean'), R2_std=('R2', 'std'), n_seeds=('R2', 'size'),
            MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
            Pearson_mean=('Pearson', 'mean'), Pearson_std=('Pearson', 'std'),
            n_train=('n_train', 'first'),
        ).reset_index().sort_values('fraction')
        # 95% CI via a t-multiplier -- see §4 for why not a fixed 1.96.
        tcrit = stats.t.ppf(0.975, m_df['n_seeds'] - 1)
        m_df['R2_ci'] = tcrit * m_df['R2_std'] / np.sqrt(m_df['n_seeds'])
        m_df['MAE_ci'] = tcrit * m_df['MAE_std'] / np.sqrt(m_df['n_seeds'])
        m_df['Pearson_ci'] = tcrit * m_df['Pearson_std'] / np.sqrt(m_df['n_seeds'])

        x = m_df['fraction'].map(x_pos)
        color = MODEL_COLORS[model_name]

        ax_r2 = axes[0, col]
        ax_r2.plot(x, m_df['R2_mean'], marker='o', label=model_name, color=color)
        ax_r2.fill_between(x, m_df['R2_mean'] - m_df['R2_ci'], m_df['R2_mean'] + m_df['R2_ci'],
                           alpha=0.25, color=color)

        ax_pearson = axes[1, col]
        ax_pearson.plot(x, m_df['Pearson_mean'], marker='o', label=model_name, color=color)
        ax_pearson.fill_between(x, m_df['Pearson_mean'] - m_df['Pearson_ci'], m_df['Pearson_mean'] + m_df['Pearson_ci'],
                                alpha=0.25, color=color)

        ax_mae = axes[2, col]
        ax_mae.plot(x, m_df['MAE_mean'], marker='o', label=model_name, color=color)
        ax_mae.fill_between(x, m_df['MAE_mean'] - m_df['MAE_ci'], m_df['MAE_mean'] + m_df['MAE_ci'],
                            alpha=0.25, color=color)

    tick_pos = [x_pos[f] for f in frac_order]
    tick_labels = []
    for f in frac_order:
        n = int(ep_df[ep_df['fraction'] == f]['n_train'].iloc[0])
        pct = n / full_n * 100
        tick_labels.append(f'{n}\n({pct:.0f}%)')

    for row in (0, 1, 2):
        ax = axes[row, col]
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_labels, fontsize=7)
        ax.set_title(ENDPOINT_LABELS.get(ep, ep), loc='center')
        ax.set_xlabel('n_train (% of training pool)')
        # Seeds/point note on every subplot -- not just col 0 -- so each panel is
        # self-contained when read/exported individually.
        ax.set_title(f'n={N_SEEDS_ZOOM} seeds/point', loc='left', fontsize=7)

    axes[0, col].axhline(0, color='grey', linewidth=0.8, linestyle='--')
    axes[0, col].set_ylabel('R² (test)')
    axes[0, col].grid(alpha=0.3)

    axes[1, col].set_ylabel('Pearson r (test)')
    axes[1, col].grid(alpha=0.3)

    axes[2, col].set_ylabel(f'MAE (test) [{ENDPOINT_LABELS.get(ep, ep)}]')
    axes[2, col].grid(alpha=0.3)

    # CI patch appended to every legend -- so each subplot's shading is explained without
    # relying on the reader having seen the figure suptitle.
    ci_patch = Patch(facecolor='grey', alpha=0.25, label='95% CI')
    for row in (0, 1, 2):
        handles, labels = axes[row, col].get_legend_handles_labels()
        axes[row, col].legend(handles=handles + [ci_patch], fontsize=8)

fig.suptitle(zoom_title, fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES}/quantity_scan_combined.png', dpi=150, bbox_inches='tight')
plt.show()